In [2]:
import pandas as pd
from lightgbm import LGBMClassifier

In [3]:
# Show complete DataFrames in notebook output.
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)

In [4]:
model_data = pd.read_parquet("/Users/abhijeetsinghparihar/Desktop/Projects/Supply Chain Project/CartIQ/data/processed/model_data.parquet")

In [5]:
print(model_data.shape)
print(model_data["target"].value_counts())

(5334991, 23)
target
0    5124089
1     210902
Name: count, dtype: int64


In [6]:
model_data.head(10)

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,cart_size,product_id,target,user_total_orders,user_avg_basket_size,user_avg_days_between_orders,user_avg_order_hour,user_product_orders,user_product_reorders,user_product_first_order,user_product_last_order,user_product_reorder_rate,product_name,aisle_id,department_id,has_bought_before,has_reordered
0,1,112108,4,4,10,9.0,4,40706,0,3,7.0,11.0,15.0,0.0,0.0,0.0,0.0,0.0,Organic Grape Tomatoes,123,4,0,0
1,1,112108,4,4,10,9.0,4,24964,0,3,7.0,11.0,15.0,0.0,0.0,0.0,0.0,0.0,Organic Garlic,83,4,0,0
2,1,112108,4,4,10,9.0,4,27845,0,3,7.0,11.0,15.0,0.0,0.0,0.0,0.0,0.0,Organic Whole Milk,84,16,0,0
3,1,112108,4,4,10,9.0,4,44359,0,3,7.0,11.0,15.0,2.0,1.0,1.0,2.0,0.5,Organic Small Bunch Celery,83,4,1,1
4,1,112108,4,4,10,9.0,4,47626,0,3,7.0,11.0,15.0,0.0,0.0,0.0,0.0,0.0,Large Lemon,24,4,0,0
5,1,112108,4,4,10,9.0,4,5707,0,3,7.0,11.0,15.0,2.0,1.0,2.0,3.0,0.5,Dark Chocolate Cinnamon Pecan Bar,3,19,1,1
6,1,112108,4,4,10,9.0,4,21903,0,3,7.0,11.0,15.0,0.0,0.0,0.0,0.0,0.0,Organic Baby Spinach,123,4,0,0
7,1,112108,4,4,10,9.0,4,45007,0,3,7.0,11.0,15.0,0.0,0.0,0.0,0.0,0.0,Organic Zucchini,83,4,0,0
8,1,112108,4,4,10,9.0,4,42001,0,3,7.0,11.0,15.0,1.0,0.0,1.0,1.0,0.0,"Tuna Ventresca, in Olive Oil",95,15,1,0
9,1,112108,4,4,10,9.0,4,21137,0,3,7.0,11.0,15.0,0.0,0.0,0.0,0.0,0.0,Organic Strawberries,24,4,0,0


In [7]:
features = [
    # Context
    "order_number",
    "order_dow",
    "order_hour_of_day",
    "days_since_prior_order",
    "cart_size",

    # User
    "user_total_orders",
    "user_avg_basket_size",
    "user_avg_days_between_orders",
    "user_avg_order_hour",

    # User + Product
    "user_product_orders",
    "user_product_reorders",
    "user_product_first_order",
    "user_product_last_order",
    "user_product_reorder_rate",

    # Product
    "product_id",
    "aisle_id",
    "department_id",

    # Additional signals
    "has_bought_before",
    "has_reordered"
]

In [8]:
X = model_data[features]
y = model_data["target"]

In [9]:
print(X.shape)
print(y.shape[0])

(5334991, 19)
5334991


In [10]:
X.isna().sum().sort_values(ascending=False)

order_number                    0
user_product_reorders           0
has_bought_before               0
department_id                   0
aisle_id                        0
product_id                      0
user_product_reorder_rate       0
user_product_last_order         0
user_product_first_order        0
user_product_orders             0
order_dow                       0
user_avg_order_hour             0
user_avg_days_between_orders    0
user_avg_basket_size            0
user_total_orders               0
cart_size                       0
days_since_prior_order          0
order_hour_of_day               0
has_reordered                   0
dtype: int64

In [11]:
from sklearn.model_selection import train_test_split

order_ids = model_data["order_id"].unique()

train_orders, valid_orders = train_test_split(
    order_ids,
    test_size=0.2,
    random_state=42
)

In [12]:
train_mask = model_data["order_id"].isin(train_orders)
valid_mask = model_data["order_id"].isin(valid_orders)

In [13]:
X_train = model_data.loc[train_mask, features]
y_train = model_data.loc[train_mask, "target"]

X_valid = model_data.loc[valid_mask, features]
y_valid = model_data.loc[valid_mask, "target"]

In [14]:
print("Train:", X_train.shape)
print("Validation:", X_valid.shape)

print("Train positive:", y_train.sum())
print("Validation positive:", y_valid.sum())

Train: (4268436, 19)
Validation: (1066555, 19)
Train positive: 169043
Validation positive: 41859


In [15]:
model = LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1
)

In [16]:
model.fit(
    X_train,
    y_train
)

[LightGBM] [Info] Number of positive: 169043, number of negative: 4099393
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.258876 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1848
[LightGBM] [Info] Number of data points in the train set: 4268436, number of used features: 19
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.039603 -> initscore=-3.188441
[LightGBM] [Info] Start training from score -3.188441


,learning_rate,0.05
,n_estimators,300
,objective,'binary'
,random_state,42
,n_jobs,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,subsample_for_bin,200000
,class_weight,None
,min_split_gain,0.0


In [17]:
valid_predictions = model.predict_proba(X_valid)[:, 1]

In [18]:
validation_results = model_data.loc[
    valid_mask,
    [
        "order_id",
        "user_id",
        "cart_size",
        "product_id",
        "target"
    ]
].copy()

validation_results["score"] = valid_predictions

In [19]:
top_10 = (
    validation_results
    .sort_values(
        ["order_id", "cart_size", "score"],
        ascending=[True, True, False]
    )
    .groupby(["order_id", "cart_size"])
    .head(10)
    .copy()
)

In [20]:
# Calculate Precision@10 and Recall@10

evaluation = (
    top_10
    .groupby(["order_id", "cart_size"])
    .agg(
        hits=("target", "sum"),
        recommended=("product_id", "count")
    )
    .reset_index()
)

# Number of products that were actually supposed to be recommended
actual_products = (
    validation_results
    .groupby(["order_id", "cart_size"])["target"]
    .sum()
    .reset_index(name="actual")
)

evaluation = evaluation.merge(
    actual_products,
    on=["order_id", "cart_size"],
    how="left"
)

# Precision@10
evaluation["precision_at_10"] = (
    evaluation["hits"] / evaluation["recommended"]
)

# Recall@10
evaluation["recall_at_10"] = (
    evaluation["hits"] / evaluation["actual"]
)

print("Precision@10:", evaluation["precision_at_10"].mean())
print("Recall@10:", evaluation["recall_at_10"].mean())

Precision@10: 0.10666586258191615
Recall@10: 0.6364899141252193


In [21]:
precision_at_10 = evaluation["precision_at_10"].mean()
recall_at_10 = evaluation["recall_at_10"].mean()

print(f"Precision@10 : {precision_at_10:.4f}")
print(f"Recall@10    : {recall_at_10:.4f}")

Precision@10 : 0.1067
Recall@10    : 0.6365


In [22]:
# Calculate Precision@5 and Recall@5

top_5 = (
    validation_results
    .sort_values(
        ["order_id", "cart_size", "score"],
        ascending=[True, True, False]
    )
    .groupby(["order_id", "cart_size"])
    .head(5)
)

evaluation_at_5 = (
    top_5
    .groupby(["order_id", "cart_size"])
    .agg(
        hits=("target", "sum"),
        recommended=("product_id", "count")
    )
    .reset_index()
)

actual_products_at_5 = (
    validation_results
    .groupby(["order_id", "cart_size"])["target"]
    .sum()
    .reset_index(name="actual")
)

evaluation_at_5 = evaluation_at_5.merge(
    actual_products_at_5,
    on=["order_id", "cart_size"],
    how="left"
)

evaluation_at_5["precision_at_5"] = (
    evaluation_at_5["hits"] / evaluation_at_5["recommended"]
)

evaluation_at_5["recall_at_5"] = (
    evaluation_at_5["hits"] / evaluation_at_5["actual"].replace(0, pd.NA)
)

print(f"Precision@5: {evaluation_at_5['precision_at_5'].mean():.4f}")
print(f"Recall@5:    {evaluation_at_5['recall_at_5'].mean():.4f}")

Precision@5: 0.1425
Recall@5:    0.4360
